In [4]:
import pandas as pd  
import numpy as np  
import re

### Подготовка объектов и признаков

- Работаем с таблицей `data` (копируем ее)
- Сортируем для удобства:

In [38]:
objects = data.copy()

objects = objects.sort_values([
    'student_id', 
    'discipline', 
    'academic_year', 
    'module'
])

Соединяем все таблицы в одну по одинаковым столбцам: 

- Далее будем создавать доп. признаки и формировать финальные объекты, на которых будем обучать модель

In [39]:
df = pd.merge(objects, students, how='left', on='student_id')

df = pd.merge(df, sop, how='left', on=['program', 'course', 'education_level', 'academic_year', 'module', 'discipline'])
df.head()

,student_id,course,exam_type,grade_10,absence_status,module,academic_year,discipline,program,education_level,place_type,difficulty_avg_score,std_deviation,students_responsed_ratio
0,1,4,Первая сдача,7,\N,2,2023,Альтернативные языки для JVM,Прикладная математика и информатика,Бакалавриат,Бюджетные,NaN,NaN,NaN
1,1,4,Первая сдача,9,\N,3,2024,Анализ программ,Прикладная математика и информатика,Бакалавриат,Бюджетные,4.0000,0.81650,0.750000
2,1,4,Первая сдача,8,\N,2,2023,Базы данных,Прикладная математика и информатика,Бакалавриат,Бюджетные,NaN,NaN,NaN
3,1,4,Первая сдача,9,\N,1,2024,Веб-поиск и ранжирование,Прикладная математика и информатика,Бакалавриат,Бюджетные,4.5000,0.80623,0.666667
4,1,4,Первая сдача,7,\N,2,2024,Веб-поиск и ранжирование,Прикладная математика и информатика,Бакалавриат,Бюджетные,4.0909,1.31111,0.733333


Создаем лаги: 

* информацию о среднем балле конкретного студента по всем дисциплинам за предыдущие модули (если данные имеются)
* минимальная полученная оценка за весь период обучения
* максимальная полученная оценка за весь период обучения
* предыдущие оценки по данной дисциплине, а также уточняющая информация (какая попытка экзамена, средняя сложность предмета в данном модуле и тп.)

In [40]:
total = df.copy()
total['grade_10'] = pd.to_numeric(total['grade_10'], errors='coerce')

- Определим порядок в столбце `exam_type` и отсортируем таблицу:

In [41]:
total['exam_type'].unique()

array(['Первая сдача', 'Пересдача', 'Пересдача с комиссией',
       'Пересдача по уважительной причине'], dtype=object)

In [42]:
total['exam_type'] = pd.Categorical(total['exam_type'], categories=['Первая сдача','Пересдача по уважительной причине','Пересдача','Пересдача с комиссией'], ordered=True)

total = total.sort_values(['student_id','academic_year','module','exam_type','discipline']).reset_index(drop=True)

Группируем по `student_id` для нахождения среднего балла, минимальной и максимальной оценки, а для добавления последних оценок группируем по `student_id` и `discipline`:

In [43]:
g_student = total.groupby('student_id')
g_subject = total.groupby(['student_id', 'discipline'])

total['grade_k-1'] = g_subject['grade_10'].shift(1)

total['module_k-1'] = g_subject['module'].shift(1)

total['difficulty_k-1'] = g_subject['difficulty_avg_score'].shift(1)

total['students_responsed_ratio_k-1'] = g_subject['students_responsed_ratio'].shift(1)

total['std_deviation_k-1'] = g_subject['std_deviation'].shift(1)

total['exam_type_k-1'] = g_subject['exam_type'].shift(1)

In [44]:
total['cnt_prev'] = g_student.cumcount()

total['sum_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cumsum())

total['min_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cummin())

total['max_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cummax())

total['avg_grade_prev'] = total['sum_prev'] / total['cnt_prev']

total = total[total['cnt_prev'] > 0].copy()

### Target


> Предсказывать будем оценку, которую студент получит в следующем модуле по дисциплине 

- Необходимо создать для каждого объекта `target` (оценка в следующий раз): 

In [45]:
total['target_grade'] = g_subject['grade_10'].shift(-1)
total['target_type'] = g_subject['exam_type'].shift(-1)
total = total[~total['target_grade'].isna()].copy()


In [46]:
total.columns

Index(['student_id', 'course', 'exam_type', 'grade_10', 'absence_status',
       'module', 'academic_year', 'discipline', 'program', 'education_level',
       'place_type', 'difficulty_avg_score', 'std_deviation',
       'students_responsed_ratio', 'grade_k-1', 'module_k-1', 'difficulty_k-1',
       'students_responsed_ratio_k-1', 'std_deviation_k-1', 'exam_type_k-1',
       'cnt_prev', 'sum_prev', 'min_prev', 'max_prev', 'avg_grade_prev',
       'target_grade', 'target_type'],
      dtype='object')

In [47]:
cols = [
    'student_id',
    'program', 'education_level', 'academic_year', 'place_type', 'course', 'absence_status', 'discipline',
    'module', 'exam_type', 'grade_10', 'difficulty_avg_score', 'std_deviation','students_responsed_ratio',
    'module_k-1', 'exam_type_k-1','grade_k-1', 'difficulty_k-1', 'std_deviation_k-1', 'students_responsed_ratio_k-1',
    
    'avg_grade_prev',
    'min_prev','max_prev', 'target_grade', 'target_type'
    ]

total = total.reindex(columns=cols)
total.head()

,student_id,program,education_level,academic_year,place_type,course,absence_status,discipline,module,exam_type,...,exam_type_k-1,grade_k-1,difficulty_k-1,std_deviation_k-1,students_responsed_ratio_k-1,avg_grade_prev,min_prev,max_prev,target_grade,target_type
13,1,Прикладная математика и информатика,Бакалавриат,2024,Бюджетные,4,\N,Веб-поиск и ранжирование,1,Первая сдача,...,NaN,NaN,NaN,NaN,NaN,7.384615,4.0,10.0,7.0,Первая сдача
17,1,Прикладная математика и информатика,Бакалавриат,2024,Бюджетные,4,\N,Правовая грамотность,2,Первая сдача,...,NaN,NaN,NaN,NaN,NaN,7.647059,4.0,10.0,8.0,Первая сдача
25,2,Прикладной анализ данных и искусственный интел...,Бакалавриат,2022,Коммерческие за счет средств вуза,4,\N,Дискретная математика,1,Первая сдача,...,NaN,NaN,NaN,NaN,NaN,7.000000,7.0,7.0,7.0,Первая сдача
27,2,Прикладной анализ данных и искусственный интел...,Бакалавриат,2022,Коммерческие за счет средств вуза,4,\N,Алгоритмы и структуры данных,2,Первая сдача,...,NaN,NaN,NaN,NaN,NaN,6.666667,6.0,7.0,7.0,Первая сдача
29,2,Прикладной анализ данных и искусственный интел...,Бакалавриат,2022,Коммерческие за счет средств вуза,4,\N,Дискретная математика,2,Первая сдача,...,Первая сдача,7.0,NaN,NaN,NaN,6.200000,5.0,7.0,7.0,Первая сдача


In [48]:
total.to_csv('total_laggs.csv', index=False)

In [49]:
total.shape

(3360, 25)

Теперь создадим объекты, но без подробной информации о предыдущих оценках: 

In [50]:
total1 = df.copy()

total1['grade_10'] = pd.to_numeric(total1['grade_10'], errors='coerce')
total1['exam_type'] = pd.Categorical(total1['exam_type'], categories=['Первая сдача','Пересдача по уважительной причине','Пересдача','Пересдача с комиссией'], ordered=True)

total1 = total1.sort_values(['student_id', 'academic_year', 'module', 'exam_type','discipline']).reset_index(drop=True)

In [51]:
g_student = total1.groupby('student_id')
g_subject = total1.groupby(['student_id', 'discipline'])

total1['grade_k-1'] = g_subject['grade_10'].shift(1)

total1['exam_type_k-1'] = g_subject['exam_type'].shift(1)

total1['cnt_prev'] = g_student.cumcount()

total1['sum_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cumsum())

total1['min_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cummin())

total1['max_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cummax())

total1['avg_grade_prev'] = total1['sum_prev'] / total1['cnt_prev']

total1 = total1[total1['cnt_prev'] > 0].copy()

In [52]:
total1['target_grade'] = g_subject['grade_10'].shift(-1)
total1['target_type'] = g_subject['exam_type'].shift(-1)
total1 = total1[~total1['target_grade'].isna()].copy()

In [53]:
total1 = total1.drop(columns=['cnt_prev', 'sum_prev'])

In [54]:
cols1 = [
    'student_id',
    'program', 'education_level', 'academic_year', 'place_type', 'course', 'absence_status', 'discipline',
    'module', 'exam_type', 'grade_10', 'difficulty_avg_score', 'std_deviation','students_responsed_ratio',
    'exam_type_k-1','grade_k-1',
    
    'avg_grade_prev',
    'min_prev','max_prev', 'target_grade', 'target_type'
    ]

total1 = total1.reindex(columns=cols1)

In [55]:
total1.to_csv('total_less_laggs.csv', index=False)

#### Проверка:

In [56]:
total1[ total1['student_id'] == 240 ][['course', 'module', 'discipline', 'exam_type', 'grade_10', 'difficulty_avg_score', 'target_grade', 'target_type']]

,course,module,discipline,exam_type,grade_10,difficulty_avg_score,target_grade,target_type
6588,2,1,Дискретная математика,Первая сдача,6,NaN,7.0,Первая сдача
6591,2,2,Алгоритмы и структуры данных,Первая сдача,5,4.1579,6.0,Первая сдача
6592,2,2,Дискретная математика,Первая сдача,7,4.6111,8.0,Первая сдача
6593,2,2,История России,Первая сдача,8,4.3333,8.0,Первая сдача
6594,2,2,История России,Первая сдача,8,3.6667,8.0,Первая сдача
6595,2,2,Математический анализ 1,Первая сдача,5,NaN,6.0,Первая сдача
6596,2,2,Основы и методология программирования,Первая сдача,4,NaN,6.0,Первая сдача
6597,2,2,Язык программирования C++,Первая сдача,5,NaN,7.0,Первая сдача


In [57]:
total1.shape

(3360, 21)